<a href="https://colab.research.google.com/github/SIVAPRASAD03/Quantum/blob/main/QuantumX_Assignment3_Grover_Pattern_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Pattern Search using Grover's Algorithm

This Colab notebook searches all eight 3-bit patterns (`000` through `111`) for a user-selected target. The oracle is built dynamically, so the same program works for any valid 3-bit pattern.

## Step 1 — import the required libraries
Run this cell once in a new Google Colab notebook.

In [4]:
!pip -q install qiskit qiskit-aer matplotlib pylatexenc

import math
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
print('Qiskit and AerSimulator are ready.')

Qiskit and AerSimulator are ready.


## Step 2 — Choose the pattern to search
Enter any three-character binary pattern. For this assignment, enter `101`. You may later rerun this cell with `000`, `011`, or `110` without changing the oracle code.

In [ ]:
target_pattern = input('Enter the target 3-bit pattern (for example, 101): ').strip()

if len(target_pattern) != 3 or any(bit not in '01' for bit in target_pattern):
    raise ValueError('Please enter exactly three binary digits, such as 101.')

search_space = [format(i, '03b') for i in range(8)]
print('Search space:', search_space)
print('Selected target:', target_pattern)

## Step 3 — Build the dynamic oracle and Grover diffuser
Qiskit writes a three-qubit state as `|q2 q1 q0⟩`. Therefore the rightmost target bit belongs to `q0`. The oracle temporarily changes every target bit equal to `0` into `1`, applies a controlled phase flip to `|111⟩`, and then restores those qubits. This makes the oracle independent of the chosen target.

In [ ]:
def apply_dynamic_oracle(qc, target):
    """Flip the phase only of the computational state named by target."""
    # Reverse target because q0 corresponds to the rightmost displayed bit.
    reversed_target = target[::-1]

    # Map the requested target state to |111>.
    for qubit, bit in enumerate(reversed_target):
        if bit == '0':
            qc.x(qubit)

    # Implement CCZ using H-CCX-H. This adds a -1 phase only to |111>.
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)

    # Undo the temporary mapping.
    for qubit, bit in enumerate(reversed_target):
        if bit == '0':
            qc.x(qubit)


def apply_diffuser(qc):
    """Apply Grover's inversion-about-the-mean operation."""
    qc.h([0, 1, 2])
    qc.x([0, 1, 2])
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])

## Step 4 — Construct and run the Grover search circuit
There are `N = 8` possible states and one marked solution. The near-optimal number of Grover iterations is `round((π/4)√N) = 2`.

In [ ]:
num_qubits = 3
num_solutions = 1
iterations = round((math.pi / 4) * math.sqrt((2**num_qubits) / num_solutions))

qc = QuantumCircuit(num_qubits, num_qubits)

# Superposition: create an equal amplitude for all eight bit patterns.
qc.h([0, 1, 2])
qc.barrier()

for _ in range(iterations):
    apply_dynamic_oracle(qc, target_pattern)
    qc.barrier()
    apply_diffuser(qc)
    qc.barrier()

# Measure q0, q1, q2 into c0, c1, c2. Counts display as c2c1c0.
qc.measure([0, 1, 2], [0, 1, 2])

print(f'Grover iterations used: {iterations}')
print(qc.draw(output='text'))

## Step 5 — Simulate measurements and display the histogram
The highest-count bit string is the pattern identified by the quantum search.

In [ ]:
shots = 2048
compiled_qc = transpile(qc, simulator)
result = simulator.run(compiled_qc, shots=shots).result()
counts = result.get_counts()
identified_pattern = max(counts, key=counts.get)

print('Measurement counts:', counts)
print(f'Pattern identified by the quantum algorithm: {identified_pattern}')
print(f'User-selected target pattern: {target_pattern}')

plot_histogram(counts, title='Grover Quantum Pattern Search Measurement Histogram')
plt.show()

if identified_pattern == target_pattern:
    print('Success: the amplified state matches the selected target.')
else:
    print('Run the circuit again; simulator measurement outcomes are probabilistic.')